# **Regression: XGBoost Regressor (eXtreme Gradient Boosting)**

## **Justification of Preprocessing Strategy**

### **Scale Invariance**
Like its cousin the Gradient Boosting Regressor, XGBoost relies on an ensemble of decision trees. Since the underlying base estimators map feature space logic via threshold splits, the algorithm remains inherently invariant to feature scale. Operations such as Standardization or Normalization will not impact the final structural splits nor improve predictive performance. Thus, to maintain efficiency and interpretability, the model will be trained on the **Original, Unscaled Data**.

### **The Boosting Philosophy: Regulated Weak Learners**
XGBoost builds trees sequentially, applying gradient descent techniques to minimize the loss function of prior iterations. To prevent instantaneous overfitting, the algorithm depends on shallow trees (Weak Learners). A depth (`max_depth`) of 3 to 6 is typically optimal. Supplying the algorithm with highly complex trees immediately nullifies its learning curve. Hence, we rely solely on internal hyperparameter optimization to define the boosting architecture.

## **Experiment Design**

We designed a robust tournament consisting of 3 optimization levels. Because sequential boosting easily memorizes data when unconstrained, we deliberately log **both Train and Test metrics (RMSE, MAE, R²)** to continuously track model generalization.

* **Baseline**: Executes XGBoost with initial recommended defaults (`n_estimators=100`, `learning_rate=0.1`, `max_depth=3`).
* **GridSearchCV**: A conservative 3-fold cross-validated search checking standard interaction points between ensemble size (`n_estimators`), speed (`learning_rate`), and depth (`max_depth`). 
* **Optuna Optimization**: Bayesian optimization applied to refine the critical parameters along a continuous space, strictly minimizing the validation RMSE on a localized search region to yield the ultimate champion.

In [1]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Regression_XGBoost")

# 2. Data Loading and Preparation
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")

categorical_cols = [
    'gender', 'ethnicity', 'smoking_status', 'education_level',
    'employment_status', 'age_groups', 'weight_status', 'income_level'
]

# Apply One-Hot Encoding
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features and target 

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage", "diabetes_risk_score"], axis=1)
y = df_final['diabetes_risk_score']

# Split data (80/20) - No stratify needed for continuous regression targets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

SEED = 42

def log_regression_metrics(y_tr_true, y_tr_pred, y_te_true, y_te_pred, duration):
    # fLogs Train and Test metrics explicitly to monitor the Overfitting Gap
    # Train Partition Metrics
    mlflow.log_metric("rmse_train", mean_squared_error(y_tr_true, y_tr_pred) ** 0.5)
    mlflow.log_metric("mae_train", mean_absolute_error(y_tr_true, y_tr_pred))
    mlflow.log_metric("r2_train", r2_score(y_tr_true, y_tr_pred))
    
    # Test Partition Metrics
    mlflow.log_metric("rmse_test", mean_squared_error(y_te_true, y_te_pred) ** 0.5)
    mlflow.log_metric("mae_test", mean_absolute_error(y_te_true, y_te_pred))
    mlflow.log_metric("r2_test", r2_score(y_te_true, y_te_pred))
    
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: BASELINE
# ---------------------------------------------------------
with mlflow.start_run(run_name="XGB_Reg_Baseline"):
    reg_base = XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=SEED,
        n_jobs=-1
    )
    
    start_time = time.time()
    reg_base.fit(X_train, y_train)
    duration = time.time() - start_time
    
    # Explicit Predictions
    y_pred_train_base = reg_base.predict(X_train)
    y_pred_test_base = reg_base.predict(X_test)
    
    mlflow.log_params(reg_base.get_params())
    mlflow.log_param("optimization", "none_default")
    
    log_regression_metrics(y_train, y_pred_train_base, y_test, y_pred_test_base, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV
# ---------------------------------------------------------
with mlflow.start_run(run_name="XGB_Reg_GridSearch"):
    param_grid = {
        "n_estimators": [50, 100],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5]
    }

    grid_reg = GridSearchCV(
        estimator=XGBRegressor(random_state=SEED, n_jobs=-1),
        param_grid=param_grid,
        cv=KFold(n_splits=3, shuffle=True, random_state=SEED),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    start_time = time.time()
    grid_reg.fit(X_train, y_train)
    duration = time.time() - start_time

    best_xgb_grid = grid_reg.best_estimator_
    
    # Explicit Predictions
    y_pred_train_grid = best_xgb_grid.predict(X_train)
    y_pred_test_grid = best_xgb_grid.predict(X_test)

    mlflow.log_params(grid_reg.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    
    log_regression_metrics(y_train, y_pred_train_grid, y_test, y_pred_test_grid, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective_reg(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 250),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 7)
    }

    model = XGBRegressor(**params, random_state=SEED, n_jobs=-1)
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=KFold(n_splits=3, shuffle=True, random_state=SEED),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    return -scores.mean()

with mlflow.start_run(run_name="XGB_Reg_Optuna"):
    study_reg = optuna.create_study(direction="minimize")
    
    start_time = time.time()
    study_reg.optimize(objective_reg, n_trials=15)
    duration = time.time() - start_time

    best_xgb_optuna = XGBRegressor(**study_reg.best_params, random_state=SEED, n_jobs=-1)
    best_xgb_optuna.fit(X_train, y_train)
    
    # Explicit Predictions
    y_pred_train_optuna = best_xgb_optuna.predict(X_train)
    y_pred_test_optuna = best_xgb_optuna.predict(X_test)

    mlflow.log_params(study_reg.best_params)
    mlflow.log_param("optimization", "optuna")
    
    log_regression_metrics(y_train, y_pred_train_optuna, y_test, y_pred_test_optuna, duration)


[I 2026-05-22 14:22:08,467] A new study created in memory with name: no-name-881a5614-4863-4aff-a347-bb827ef71b54
[I 2026-05-22 14:22:10,707] Trial 0 finished with value: 0.9162388267883501 and parameters: {'n_estimators': 231, 'learning_rate': 0.031341779769598614, 'max_depth': 3}. Best is trial 0 with value: 0.9162388267883501.
[I 2026-05-22 14:22:16,129] Trial 1 finished with value: 0.3615941372394593 and parameters: {'n_estimators': 228, 'learning_rate': 0.08982806891819212, 'max_depth': 7}. Best is trial 1 with value: 0.3615941372394593.
[I 2026-05-22 14:22:19,749] Trial 2 finished with value: 1.5967611592693187 and parameters: {'n_estimators': 110, 'learning_rate': 0.02041716956274073, 'max_depth': 7}. Best is trial 1 with value: 0.3615941372394593.
[I 2026-05-22 14:22:20,947] Trial 3 finished with value: 0.5517132493784055 and parameters: {'n_estimators': 60, 'learning_rate': 0.1553378361547532, 'max_depth': 4}. Best is trial 1 with value: 0.3615941372394593.
[I 2026-05-22 14:22

## Winner Run Selection (Priority Elimination Framework)

### Policy
A run is only eligible to win if it does NOT show evidence of overfitting or underfitting. Before applying the MAE/RMSE/R² decision rules, we require the Train→Test gaps to remain small enough to indicate acceptable generalization. Runs with unstable generalization are disqualified regardless of metric rank.

### Selection Criteria (priority order)
1. **Generalization filter (mandatory):** runs with overfitting or underfitting are removed from consideration.
2. **Priority 1 (60%): Lowest MAE (Test)** — primary objective for regression accuracy.
3. **Priority 2 (30%): Lowest RMSE (Test)** — used to reject runs where RMSE grows disproportionately relative to MAE.
4. **Priority 3 (10%): Acceptable R² (Test)** — confirms explanatory quality.
5. **Tiebreaker: Lowest Fit Time** — if MAE, RMSE, and R² are effectively tied.

### Runs Summary

| Run | MAE (Train) | MAE (Test) | RMSE (Train) | RMSE (Test) | R² (Train) | R² (Test) | Fit Time |
|---|---:|---:|---:|---:|---:|---:|---:|
| XGB_Reg_Baseline | 0.43519 | 0.45143 | 0.56692 | 0.58906 | 0.99608 | 0.99580 | 1.62s |
| XGB_Reg_GridSearch | 0.27693 | 0.30077 | 0.35181 | 0.38785 | 0.99849 | 0.99818 | 13.88s |
| XGB_Reg_Optuna | 0.20119 | 0.23575 | 0.25589 | 0.30536 | 0.99920 | 0.99887 | 42.61s |

### Generalization Check (Test − Train)
- **XGB_Reg_Baseline:** MAE gap = 0.45143 − 0.43519 = **+0.01624** and RMSE gap = 0.58906 − 0.56692 = **+0.02215** → PASS.
- **XGB_Reg_GridSearch:** MAE gap = 0.30077 − 0.27693 = **+0.02384** and RMSE gap = 0.38785 − 0.35181 = **+0.03605** → PASS.
- **XGB_Reg_Optuna:** MAE gap = 0.23575 − 0.20119 = **+0.03457** and RMSE gap = 0.30536 − 0.25589 = **+0.04947** → PASS.

### Overfitting / Underfitting Validation
- **Overfitting check:** no run shows a catastrophic Train/Test divergence. All runs keep small and controlled MAE/RMSE gaps.
- **Underfitting check:** no run shows weak predictive power; all Test R² values are very high (>0.995), so none is underfitting.
- **RMSE vs MAE stability:** RMSE remains consistent relative to MAE in all runs, with no evidence of catastrophic error spikes.

### Step-by-Step Elimination
**Step 1 — Apply the generalization filter**
- Passing runs: all three runs.

**Step 2 — Compare Test MAE (Priority 1 — 60%)**
- XGB_Reg_Optuna: 0.23575
- XGB_Reg_GridSearch: 0.30077
- XGB_Reg_Baseline: 0.45143
- Lowest MAE: **XGB_Reg_Optuna**.

**Step 3 — Compare Test RMSE (Priority 2 — 30%)**
- XGB_Reg_Optuna also has the lowest RMSE (0.30536), confirming stability.

**Step 4 — Check Test R² (Priority 3 — 10%)**
- XGB_Reg_Optuna has the highest Test R² (0.99887).

### Final Decision
**Winner: XGB_Reg_Optuna**

**Justification:** `XGB_Reg_Optuna` is valid under generalization checks and dominates the three regression criteria (lowest MAE, lowest RMSE, highest R²). Fit time is not needed as a tiebreaker.

## Winner Hyperparameters
| Parameter | Value |
|---|---|
| **n_estimators** | 200 |
| **learning_rate** | 0.07300190646052784 |
| **max_depth** | 6 |
| **random_state** | 42 |
| **n_jobs** | -1 |